# Automated Weather Map API Workflow
## By: James John  

This notebook builds an automated workflow for generating U.S. choropleth maps of historical weather data using the OpenWeather API.

The pipeline produces comparative visualizations of key climate variables including:
- Temperature (°F)
- Wind speed (mph)
- Humidity (%)

Users can select two different years for comparison, enabling direct temporal analysis. The workflow also includes a difference mapping feature to visualize changes between selected years.

All data is retrieved via the OpenWeather API and requires a valid API key.

### User Inputs

In [ ]:
# Input an int for year
year1 = 
year2 =
API_key = ""

### Start of Workflow

In [1]:
# Import everything 
import pandas as pd
import requests
from datetime import datetime, timedelta
import time
import os 
import plotly.express as px

### Functions

In [ ]:
# Function to call API
def get_daily_weather(lat, lon, date):
    url = "https://api.openweathermap.org/data/3.0/onecall/day_summary"
    
    params = {
        "lat": lat,
        "lon": lon,
        "date": date,
        "appid": API_KEY,
        "units": "metric"
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print("Error:", response.status_code, response.text)
        return None
    
    return response.json()

# Lattitude and longitude of the center of each state 
state_coords = {
    "AL": (32.8, -86.8), "AK": (64.2, -149.5), "AZ": (34.3, -111.7),
    "AR": (35.1, -92.4), "CA": (36.8, -119.4), "CO": (39.0, -105.5),
    "CT": (41.6, -72.7), "DE": (39.0, -75.5), "FL": (27.8, -81.7),
    "GA": (32.7, -83.3), "HI": (19.9, -155.5), "ID": (44.2, -114.5),
    "IL": (40.0, -89.2), "IN": (40.3, -86.1), "IA": (42.1, -93.5),
    "KS": (38.5, -98.0), "KY": (37.5, -85.3), "LA": (31.2, -92.3),
    "ME": (45.3, -69.0), "MD": (39.0, -76.7), "MA": (42.4, -71.4),
    "MI": (44.3, -85.6), "MN": (46.3, -94.2), "MS": (32.7, -89.7),
    "MO": (38.5, -92.5), "MT": (46.9, -110.0), "NE": (41.5, -99.8),
    "NV": (39.5, -116.9), "NH": (43.2, -71.6), "NJ": (40.1, -74.5),
    "NM": (34.4, -106.1), "NY": (43.0, -75.0), "NC": (35.5, -79.4),
    "ND": (47.5, -100.5), "OH": (40.4, -82.8), "OK": (35.6, -97.5),
    "OR": (44.0, -120.5), "PA": (41.2, -77.2), "RI": (41.7, -71.5),
    "SC": (33.8, -80.9), "SD": (44.4, -100.2), "TN": (35.8, -86.4),
    "TX": (31.0, -100.0), "UT": (39.3, -111.7), "VT": (44.0, -72.7),
    "VA": (37.5, -78.7), "WA": (47.4, -120.7), "WV": (38.6, -80.6),
    "WI": (44.5, -89.5), "WY": (43.0, -107.5)
}

# Function to call API for chosen years
def collect_weather_year(year):
    start_date = datetime(year, 1, 1)
    end_date = datetime(year, 12, 31)

    call_count = 0
    data = []
    current_date = start_date

    while current_date <= end_date:
        date_str = current_date.strftime("%Y-%m-%d")

        for state, (lat, lon) in state_coords.items():

            result = get_daily_weather(lat, lon, date_str)

            if result:
                row = {
                    "state": state,
                    "date": date_str,
                    "year": year,
                    "wind_speed": result["wind"]["max"]["speed"],
                    "humidity": result["humidity"]["afternoon"]
                }
                data.append(row)

            call_count += 1
            time.sleep(0.5)

            if call_count % 20 == 0:
                print(f"{year}: {call_count} calls made")

        current_date += timedelta(days=5)

    print(f"{year}: {call_count} total calls made")
    return data

### Collect and Clean Data

In [ ]:
# Collect data from the 2 years
data_1 = collect_weather_year(year1)
data_2 = collect_weather_year(year2)

# Create dataframes of those 2 datasets and combine to 1 dataframe
df_1 = pd.DataFrame(data_1)
df_2 = pd.DataFrame(data_2)
df = pd.concat([df_1, df_2], ignore_index=True)

# Clean data and change units
df["date"] = pd.to_datetime(df["date"])
df["wind_speed_mph"] = df["wind_speed"] * 2.23694
df["temp_f"] = df["temp_afternoon"] * 9/5 + 32

# Seperate data into 2 dataframes
df_year1 = df[df["date"].dt.year == year1]
df_year2 = df[df["date"].dt.year == year2]

# Get the averages for each state in each year
avg_1 = df_year1.groupby("state")[["wind_speed_mph", "humidity", "temp_f"]].mean().round(2).reset_index()
avg_2 = df_year2.groupby("state")[["wind_speed_mph", "humidity", "temp_f"]].mean().round(2).reset_index()

### Create Visual Maps

In [ ]:
datasets = {
    year1: avg_year1,
    year2: avg_year2
}

plots = {
    "wind_speed_mph": {
        "title": "Average Wind Speed",
        "colorscale": "matter",
        "colorbar_title": "Wind Speed (mph)"
    },
    "humidity": {
        "title": "Average Humidity",
        "colorscale": "deep",
        "colorbar_title": "Humidity (%)"
    },
    "temp_f": {
        "title": "Average Temperature",
        "colorscale": "RdYlBu_r",
        "colorbar_title": "Temperature (°F)"
    }
}

for col, cfg in plots.items():

    for year in [year1, year2]:
        data = datasets[year]

        fig = px.choropleth(
            data,
            locations="state",
            locationmode="USA-states",
            color=col,
            scope="usa",
            color_continuous_scale=cfg["colorscale"],
            title=f"{cfg['title']} ({year})"
        )

        fig.update_layout(title_x=0.44)

        fig.update_coloraxes(
            colorbar=dict(
                title=cfg["colorbar_title"],
                title_side="top"
            )
        )

        fig.show()

### Create Differences Dataframe

In [ ]:
# Merge both dataframes to ensure states allign
diff_df = avg_year2.merge(
    avg_year1,
    on="state",
    suffixes=("_year2", "_year1")
)

# Create collumns for the differences
diff_df["wind_diff"] = diff_df["wind_speed_mph_year2"] - diff_df["wind_speed_mph_year1"]
diff_df["humidity_diff"] = diff_df["humidity_year2"] - diff_df["humidity_year1"]
diff_df["temp_diff"] = diff_df["temp_f_year2"] - diff_df["temp_f_year1"]

### Plot Differences Maps

In [ ]:
diff_plots = {
    "wind_diff": {
        "title": f"Wind Speed Change ({year2} - {year1})",
        "colorscale": "RdBu_r",
        "colorbar_title": "Δ Wind Speed (mph)"
    },
    "humidity_diff": {
        "title": f"Humidity Change ({year2} - {year1})",
        "colorscale": "RdBu_r",
        "colorbar_title": "Δ Humidity (%)"
    },
    "temp_diff": {
        "title": f"Temperature Change ({year2} - {year1})",
        "colorscale": "RdBu_r",
        "colorbar_title": "Δ Temperature (°F)"
    }
}

for col, cfg in diff_plots.items():

    fig = px.choropleth(
        diff_df,
        locations="state",
        locationmode="USA-states",
        color=col,
        scope="usa",
        color_continuous_scale=cfg["colorscale"],
        title=cfg["title"]
    )

    fig.update_layout(title_x=0.44)

    fig.update_coloraxes(
        colorbar=dict(
            title=cfg["colorbar_title"],
            title_side="top"
        )
    )

    fig.show()